# Corpus similarity (embeddings)

Loads [`output/corpus_20_resumes.csv`](output/corpus_20_resumes.csv), encodes **each row** in **Summary**, **Experience**, and **Skills** with `all-MiniLM-L6-v2`, adds an **`embedding`** column (JSON array per row), and writes **`output/corpus_with_embedding.csv`**.

Set **`TARGET_RESUME_ID`** below to choose which resume is “me” for the optional ranking cell.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd().resolve()
if (_cwd / "corpus_similarity.ipynb").exists():
    NOTEBOOK_DIR = _cwd
elif (_cwd / "peopleLikeMe" / "corpus_similarity.ipynb").exists():
    NOTEBOOK_DIR = _cwd / "peopleLikeMe"
else:
    NOTEBOOK_DIR = _cwd

OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = OUTPUT_DIR / "corpus_20_resumes.csv"
META_PATH = OUTPUT_DIR / "resume_metadata.csv"
OUT_EMB_PATH = OUTPUT_DIR / "corpus_with_embedding.csv"

TARGET_RESUME_ID = 1  # change to any ResumeID in the corpus

df = pd.read_csv(CORPUS_PATH, dtype={"ResumeID": int, "ItemID": int})
df["Title"] = df["Title"].fillna("").astype(str)

CORPUS_PATH.resolve(), len(df), sorted(df["ResumeID"].unique())[:5], "...", df["Section"].unique().tolist()

### Install sentence-transformers Scikit-learn

sentence-transformers
- A library for turning sentences or short texts into fixed-size vectors (embeddings) so you can compare meaning, not just words.
- Built on PyTorch and Hugging Face Transformers.
Pre-trained models (e.g. MiniLM, MPNet) map text to dense vectors; similar meanings tend to sit close in vector space.
Common uses: semantic search, duplicate detection, clustering, re-ranking, similarity between documents or queries.
It’s the usual choice when you want “what does this sentence mean?” style similarity without training a big model from scratch.

scikit-learn

- The main Python library for classical / tabular machine learning and scientific computing around models.

Together: sentence-transformers gives you text embeddings; scikit-learn can use those vectors (or other features) for clustering, classification, nearest neighbors, etc.

### Load embedding model
all-MiniLM-L6-v2 (from SentenceTransformers) because it’s:

- Built specifically for semantic similarity
It converts text → vectors (embeddings)
Similar meaning → vectors close together (cosine similarity)
That’s exactly what your workflow needs:
Resume bullets ↔ skills/tasks matching
- Very efficient
Small model (~80MB)
Fast on CPU (no GPU needed)
Great for batching thousands of rows (like O*NET)
- “Good enough” accuracy

Is it free?
- Yes — completely free
Open-source (Apache 2.0 license)
No API calls
Runs locally
No usage limits
No cost per embedding

- That’s why it’s commonly used in: prototypes and offline pipelines

In [ ]:
# pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded successfully")

### Encode Summary, Experience, and Skills (one embedding per row)

Education rows are kept in the export with an **empty** `embedding`. Vectors are stored as **JSON arrays** in the CSV.

In [ ]:
ENCODE_SECTIONS = {"Summary", "Experience", "Skills"}


def row_to_encode_text(row: pd.Series) -> str:
    section = str(row["Section"]).strip()
    text = str(row["Text"]).strip()
    title = str(row["Title"]).strip() if pd.notna(row["Title"]) else ""
    if section == "Summary":
        return text
    if section == "Experience":
        return f"{title}: {text}" if title else text
    if section == "Skills":
        return text
    return text


mask = df["Section"].isin(ENCODE_SECTIONS)
encode_idx = df.index[mask]
texts = [row_to_encode_text(df.loc[i]) for i in encode_idx]

vectors = model.encode(texts, show_progress_bar=True)

df_out = df.copy()
df_out["embedding"] = ""
for row_i, idx in enumerate(encode_idx):
    df_out.at[idx, "embedding"] = json.dumps(vectors[row_i].tolist())

df_out.to_csv(OUT_EMB_PATH, index=False, encoding="utf-8", na_rep="")

encoded_rows = int(mask.sum())
print("Wrote", OUT_EMB_PATH.resolve())
print("Encoded rows:", encoded_rows, "vector dim:", vectors.shape[1] if len(vectors) else 0)

### Optional: who is most like `TARGET_RESUME_ID`?

Mean-pool all encoded row vectors **per resume** (Summary + Experience + Skills only), then **cosine similarity** between that mean vector and each other resume’s mean vector.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

ids = sorted(df_out["ResumeID"].unique())
if TARGET_RESUME_ID not in ids:
    raise ValueError(f"TARGET_RESUME_ID={TARGET_RESUME_ID} not in corpus")

per_resume: dict[int, list[np.ndarray]] = {rid: [] for rid in ids}
for _, row in df_out.iterrows():
    if not row["embedding"]:
        continue
    per_resume[int(row["ResumeID"])].append(np.array(json.loads(row["embedding"]), dtype=np.float32))

pooled = {}
for rid, vecs in per_resume.items():
    if not vecs:
        continue
    pooled[rid] = np.mean(np.stack(vecs, axis=0), axis=0).reshape(1, -1)

target = pooled[TARGET_RESUME_ID]
rows = []
for rid in ids:
    if rid not in pooled:
        continue
    sim = float(cosine_similarity(target, pooled[rid])[0, 0])
    rows.append({"ResumeID": rid, "pooled_cosine_similarity": sim})

rank_df = pd.DataFrame(rows).sort_values("pooled_cosine_similarity", ascending=False)
if META_PATH.exists():
    meta = pd.read_csv(META_PATH)
    rank_df = rank_df.merge(meta, on="ResumeID", how="left")

rank_df.reset_index(drop=True)